# QCML for High-Yield Bond Clustering and Similarity
### Extending QCML Paper [Q3]: *Supervised Similarity for High-Yield Corporate Bonds*

**Context**

Corporate bonds are illiquid: most bonds trade only a few times per week, and many never trade at all on a given day. To price a bond with no recent quote, practitioners look for *comparable* bonds — instruments with similar credit profile, duration, and sector. This is the **bond similarity problem**.

Rosaler et al. (2025) showed that QCML outperforms classical tree-based models at learning similarity metrics in high-yield (HY) markets. This notebook re-implements and extends that idea using:

1. **Real market calibration**: OAS spreads by rating tier (BB, B, CCC) pulled live from FRED (Federal Reserve Economic Data — free, no API key required)
2. **Synthetic HY bond universe**: 600 bonds with 10 financial features calibrated to real spread levels, with realistic inter-feature correlations
3. **QCML clustering**: unsupervised ground-state embeddings clustered by rating and sector
4. **Similarity recovery analysis**: cosine similarity of QCML embeddings vs. ground-truth bond similarity — the core metric from [Q3]
5. **QCML feature diagnostics**: which bond characteristics drive the quantum geometry

**Bond features (10 dimensions)**

| Feature | Description |
|---|---|
| `oas_spread` | Option-adjusted spread over Treasuries (bps) |
| `duration` | Modified duration (years) |
| `ytm` | Yield to maturity (%) |
| `coupon` | Coupon rate (%) |
| `price` | Clean price (cents on dollar) |
| `time_to_maturity` | Years to maturity |
| `leverage` | Issuer net debt / EBITDA |
| `interest_coverage` | Issuer EBITDA / interest expense |
| `log_amount` | Log of face value outstanding ($M) |
| `bid_ask` | Bid-ask spread (price points) |

**References**
- [Q3] Rosaler et al., *Supervised Similarity for High-Yield Corporate Bonds with QCML*, QognitiveAI / arXiv:2502.01495, Feb 2025
- [Q2] Samson et al., *QCML: Financial Forecasting*, QognitiveAI, Sep 2024
- FRED: ICE BofA HY Index OAS series BAMLH0A1HYBB, BAMLH0A2HYB, BAMLH0A3HYC


## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# qcml/ lives in the same directory as this notebook
from qcml.core import QCML
from qcml.utils import (
    best_kmeans,
    align_labels,
    cluster_accuracy,
    cluster_metrics,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

SEED = 42
rng  = np.random.default_rng(SEED)

# Rating and sector labels
RATING_NAMES = {0: "BB", 1: "B", 2: "CCC"}
SECTOR_NAMES = {0: "Energy", 1: "Healthcare", 2: "Technology", 3: "Consumer", 4: "Telecom"}
RATING_COLORS = {0: "#2196F3", 1: "#FF9800", 2: "#F44336"}   # blue / orange / red
SECTOR_COLORS = ["#4CAF50", "#9C27B0", "#00BCD4", "#FF5722", "#607D8B"]

print("Imports OK.")

## 2. Real Market Context — FRED OAS Spreads by Rating Tier

We pull live ICE BofA OAS spread time series from FRED to anchor our synthetic bond universe to real market conditions.  
These series track the *average* spread for all BB-, B-, and CCC-rated US high-yield bonds.

Series used:
- `BAMLH0A1HYBB` — BB-rated HY bonds
- `BAMLH0A2HYB`  — B-rated HY bonds
- `BAMLH0A3HYC`  — CCC & lower rated HY bonds

In [ ]:
try:
    import pandas_datareader.data as web
    from datetime import datetime

    start, end = "2015-01-01", "2025-12-31"
    series = {
        "BB_OAS" : "BAMLH0A1HYBB",
        "B_OAS"  : "BAMLH0A2HYB",
        "CCC_OAS": "BAMLH0A3HYC",
    }
    oas_df = pd.concat(
        {name: web.DataReader(code, "fred", start, end)
         for name, code in series.items()},
        axis=1,
    )
    oas_df.columns = list(series.keys())
    oas_df = oas_df.dropna()

    # Calibration means (2020–2025 post-COVID window)
    recent = oas_df["2020-01-01":]
    oas_means = recent.mean()

    print("FRED data loaded successfully.")
    print(f"Date range: {oas_df.index[0].date()} → {oas_df.index[-1].date()}")
    print(f"\n2020–2025 mean OAS spreads (bps):")
    for k, v in oas_means.items():
        print(f"  {k:8s}: {v:.0f} bps")

    FRED_AVAILABLE = True

except Exception as e:
    print(f"FRED unavailable ({e}). Using hard-coded calibration values.")
    # 2020–2025 approximate averages from FRED
    oas_means = pd.Series({"BB_OAS": 278.0, "B_OAS": 490.0, "CCC_OAS": 952.0})
    FRED_AVAILABLE = False
    oas_df = None

In [ ]:
if FRED_AVAILABLE:
    fig, ax = plt.subplots(figsize=(12, 4))
    for col, color in zip(oas_df.columns, ["#2196F3", "#FF9800", "#F44336"]):
        ax.plot(oas_df.index, oas_df[col], label=col, color=color, linewidth=1.2, alpha=0.85)
    ax.axvspan(pd.Timestamp("2020-02-01"), pd.Timestamp("2020-05-01"),
               alpha=0.15, color="gray", label="COVID-19 shock")
    ax.set_ylabel("OAS (bps)")
    ax.set_title("ICE BofA US High-Yield OAS by Rating Tier (FRED)\n"
                 "The spread gap between BB, B, and CCC defines the clustering structure QCML learns")
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig("fred_oas_spreads.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("Skipping FRED plot (not available).")

## 3. Synthetic High-Yield Bond Universe

We generate 600 synthetic bonds (200 BB, 240 B, 160 CCC — matching the approximate composition of the HY index) with 10 features calibrated to the FRED spread levels above.

**Inter-feature correlations are realistic:**
- Higher spread → higher YTM → lower price → higher leverage → lower coverage
- CCC bonds tend to have shorter durations (refinancing risk / distressed structure)
- Bid-ask spread widens significantly for CCC (illiquidity premium)

**Sectors add a second clustering layer** (5 sectors with characteristic risk profiles).

In [ ]:
# ── Rating-level means calibrated to FRED ──────────────────────────────────
bb_oas  = float(oas_means["BB_OAS"])
b_oas   = float(oas_means["B_OAS"])
ccc_oas = float(oas_means["CCC_OAS"])

# Each row: [oas_spread, duration, ytm, coupon, price, time_to_mat, leverage, coverage, log_amount, bid_ask]
RATING_MEANS = np.array([
    [bb_oas,   5.10,  6.40, 5.80, 98.5, 6.20, 3.40, 4.60, 6.40, 0.75],   # BB
    [b_oas,    4.70,  8.50, 7.20, 92.0, 5.10, 5.10, 2.90, 6.00, 1.20],   # B
    [ccc_oas,  3.90, 13.20, 8.90, 74.0, 4.20, 7.80, 1.50, 5.50, 2.50],   # CCC
])

# Relative standard deviations per feature (cross-sectional dispersion)
FEAT_STDS = np.array([0.25, 0.12, 0.18, 0.12, 0.06, 0.20, 0.25, 0.30, 0.15, 0.30])

# Sector multipliers on [oas, leverage, price]: Energy, Healthcare, Tech, Consumer, Telecom
SECTOR_MODS = np.array([
    [1.30,  1.50, 0.95],   # Energy: wider spread, higher leverage, lower price
    [0.90,  0.80, 1.03],   # Healthcare: tighter, lower leverage, premium price
    [0.85,  0.70, 1.04],   # Technology: tightest, lowest leverage, premium
    [1.00,  1.00, 1.00],   # Consumer: baseline
    [1.10,  1.20, 0.97],   # Telecom: slightly wider, more levered
])

N_BONDS   = [200, 240, 160]   # BB, B, CCC counts
N_SECTORS = 5
D_FEAT    = 10
FEAT_NAMES = ["oas_spread", "duration", "ytm", "coupon", "price",
              "time_to_mat", "leverage", "coverage", "log_amount", "bid_ask"]

bonds, y_rating, y_sector = [], [], []

for rating_idx, n in enumerate(N_BONDS):
    base_mean = RATING_MEANS[rating_idx].copy()
    base_std  = base_mean * FEAT_STDS

    # Assign sectors roughly proportionally
    sectors = rng.integers(0, N_SECTORS, size=n)

    for i in range(n):
        s = int(sectors[i])
        mean = base_mean.copy()

        # Apply sector modifiers on oas (idx 0), leverage (idx 6), price (idx 4)
        mean[0] *= SECTOR_MODS[s, 0]
        mean[6] *= SECTOR_MODS[s, 1]
        mean[4] *= SECTOR_MODS[s, 2]

        # Correlated noise: OAS and YTM move together; price moves opposite
        noise_factor = rng.normal(0, 1)
        x = rng.normal(mean, base_std)
        x[2] += noise_factor * base_std[0] * 0.6    # ytm correlated with oas
        x[4] -= noise_factor * base_std[4] * 0.5    # price anti-correlated
        x[6] += noise_factor * base_std[6] * 0.4    # leverage correlated
        x[7] -= noise_factor * base_std[7] * 0.4    # coverage anti-correlated

        # Enforce physical constraints (no negative spreads / prices)
        x[0] = max(x[0], 50.0)    # min 50 bps spread
        x[4] = np.clip(x[4], 50.0, 115.0)   # price in [50, 115]
        x[1] = max(x[1], 0.5)    # min 0.5yr duration
        x[7] = max(x[7], 0.1)    # min coverage
        x[9] = max(x[9], 0.1)    # min bid-ask

        bonds.append(x)
        y_rating.append(rating_idx)
        y_sector.append(s)

bonds    = np.array(bonds)
y_rating = np.array(y_rating)
y_sector = np.array(y_sector)

# Shuffle
perm  = rng.permutation(len(bonds))
bonds, y_rating, y_sector = bonds[perm], y_rating[perm], y_sector[perm]

N_TOTAL = len(bonds)
print(f"Synthetic HY bond universe: {N_TOTAL} bonds × {D_FEAT} features")
print(f"Rating composition:  BB={sum(y_rating==0)}  B={sum(y_rating==1)}  CCC={sum(y_rating==2)}")
print(f"Sector composition:  {[sum(y_sector==s) for s in range(N_SECTORS)]}")

df_bonds = pd.DataFrame(bonds, columns=FEAT_NAMES)
df_bonds["rating"] = [RATING_NAMES[r] for r in y_rating]
df_bonds["sector"] = [SECTOR_NAMES[s] for s in y_sector]
df_bonds.head()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.flatten()

for i, feat in enumerate(FEAT_NAMES):
    ax = axes[i]
    for r, color in RATING_COLORS.items():
        vals = bonds[y_rating == r, i]
        ax.hist(vals, bins=25, alpha=0.55, color=color,
                label=RATING_NAMES[r], density=True)
    ax.set_title(feat, fontsize=9)
    ax.set_yticks([])
    ax.grid(True, alpha=0.2)

handles = [plt.Rectangle((0,0),1,1, color=RATING_COLORS[r]) for r in range(3)]
fig.legend(handles, [RATING_NAMES[r] for r in range(3)],
           loc="lower center", ncol=3, fontsize=9, framealpha=0.9)
fig.suptitle("Feature Distributions by Rating Tier", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("bond_feature_distributions.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = pd.DataFrame(bonds, columns=FEAT_NAMES).corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.4,
            annot_kws={"size": 8})
ax.set_title("Bond Feature Correlation Matrix\n"
             "(OAS ↔ YTM ↔ Leverage highly correlated; Price ↔ Coverage anti-correlated)")
plt.tight_layout()
plt.savefig("bond_correlation_matrix.png", bbox_inches="tight", dpi=150)
plt.show()

## 5. Preprocessing

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(bonds)   # shape (600, 10)

print(f"X shape: {X.shape}")
print(f"Mean (should be ~0): {X.mean(axis=0).round(3)}")
print(f"Std  (should be ~1): {X.std(axis=0).round(3)}")

## 6. Baseline Clustering — Raw Features and PCA

In [ ]:
K_RATING = 3   # BB / B / CCC
K_SECTOR = 5   # 5 sectors

# ── Raw K-Means ──────────────────────────────────────────────────────────────
km_raw      = best_kmeans(X, K_RATING)
acc_raw_r   = cluster_accuracy(y_rating, km_raw, K_RATING)
met_raw_r   = cluster_metrics(y_rating, km_raw, X)

# ── PCA + K-Means ────────────────────────────────────────────────────────────
pca = PCA(n_components=0.95, random_state=SEED)
X_pca     = pca.fit_transform(X)
km_pca    = best_kmeans(X_pca, K_RATING)
acc_pca_r = cluster_accuracy(y_rating, km_pca, K_RATING)
met_pca_r = cluster_metrics(y_rating, km_pca, X_pca)

print(f"Baseline — K-Means on raw features (k={K_RATING} ratings)")
print(f"  Accuracy: {acc_raw_r:.4f}")
for k, v in met_raw_r.items():
    print(f"  {k:12s}: {v:.4f}")

print(f"\nBaseline — PCA ({pca.n_components_} components) + K-Means")
print(f"  Accuracy: {acc_pca_r:.4f}")
for k, v in met_pca_r.items():
    print(f"  {k:12s}: {v:.4f}")

## 7. QCML Training

QCML is trained entirely **unsupervised** — no rating or sector labels are used.

The algorithm builds a Hamiltonian $H(x) = \sum_k (A_k - x_k I)^2$ per bond and finds the ground-state embedding $|\psi_0(x)\rangle$ on the unit sphere $S^{m-1}$.

With $D=10$ financial features and $m=16$ (Hilbert dimension), the model learns 10 symmetric $16 \times 16$ observable matrices that encode the geometry of the high-yield credit space.

In [ ]:
HILBERT_DIM = 16     # D=10 features; m=16 gives full reconstruction capacity
W           = 0.25   # variance weight — best for linear separability
EPOCHS      = 500
LR          = 3e-3
BATCH       = 32

model = QCML(hilbert_dim=HILBERT_DIM, lr=LR, w=W, seed=SEED)
model.fit(X, epochs=EPOCHS, batch_size=BATCH, verbose=True)

In [ ]:
Z = model.transform(X)   # shape (600, 16) — unit vectors on S^15

print(f"Embeddings Z: shape={Z.shape}")
print(f"All unit vectors: {np.allclose(np.linalg.norm(Z, axis=1), 1.0)}")

## 8. QCML Clustering — Rating and Sector Recovery

In [ ]:
# Rating clustering (k=3)
km_qcml_r   = best_kmeans(Z, K_RATING)
acc_qcml_r  = cluster_accuracy(y_rating, km_qcml_r, K_RATING)
met_qcml_r  = cluster_metrics(y_rating, km_qcml_r, Z)

# Sector clustering (k=5)
km_qcml_s   = best_kmeans(Z, K_SECTOR)
acc_qcml_s  = cluster_accuracy(y_sector, km_qcml_s, K_SECTOR)
met_qcml_s  = cluster_metrics(y_sector, km_qcml_s, Z)

print(f"QCML + K-Means — Rating clustering (k={K_RATING})")
print(f"  Accuracy (Hungarian): {acc_qcml_r:.4f}")
for k, v in met_qcml_r.items():
    print(f"  {k:12s}: {v:.4f}")

print(f"\nQCML + K-Means — Sector clustering (k={K_SECTOR})")
print(f"  Accuracy (Hungarian): {acc_qcml_s:.4f}")
for k, v in met_qcml_s.items():
    print(f"  {k:12s}: {v:.4f}")

## 9. Similarity Recovery Analysis

This is the core analysis of [Q3]: does QCML learn a geometry where **similar bonds are close**?

We define a ground-truth similarity matrix:
$$\text{Sim}^\text{true}_{ij} = \mathbb{1}[\text{rating}(i) = \text{rating}(j)]$$

And measure the **cosine similarity** of QCML embeddings:
$$\text{Sim}^\text{QCML}_{ij} = \langle\psi_0(x_i),\, \psi_0(x_j)\rangle$$

Since both $|\psi_0\rangle$ are unit vectors, this inner product equals the cosine similarity directly.

We compare this to the PCA baseline using the same inner product on PCA-projected features.

In [ ]:
# Use a random subsample for computational tractability
N_SIM = 200
idx   = rng.choice(N_TOTAL, size=N_SIM, replace=False)

Z_sub      = Z[idx]                       # (200, m) — unit vectors
X_pca_sub  = X_pca[idx]                   # (200, n_pca)
X_pca_norm = X_pca_sub / (np.linalg.norm(X_pca_sub, axis=1, keepdims=True) + 1e-12)
y_r_sub    = y_rating[idx]
y_s_sub    = y_sector[idx]

# Ground-truth similarity matrices
sim_true_rating = (y_r_sub[:, None] == y_r_sub[None, :]).astype(float)
sim_true_sector = (y_s_sub[:, None] == y_s_sub[None, :]).astype(float)

# Cosine similarity matrices
sim_qcml = Z_sub @ Z_sub.T                  # (200, 200)
sim_pca  = X_pca_norm @ X_pca_norm.T        # (200, 200)

# Upper-triangle vectors (exclude diagonal)
iu   = np.triu_indices(N_SIM, k=1)
s_gt_r  = sim_true_rating[iu]
s_qcml  = sim_qcml[iu]
s_pca   = sim_pca[iu]

# Mean cosine similarity: same-rating pairs vs. different-rating pairs
same_qcml = s_qcml[s_gt_r == 1].mean()
diff_qcml = s_qcml[s_gt_r == 0].mean()
same_pca  = s_pca[s_gt_r == 1].mean()
diff_pca  = s_pca[s_gt_r == 0].mean()

print("Cosine similarity — same-rating pairs vs. different-rating pairs")
print(f"  QCML:  same={same_qcml:.4f}  diff={diff_qcml:.4f}  gap={same_qcml - diff_qcml:.4f}")
print(f"  PCA:   same={same_pca:.4f}  diff={diff_pca:.4f}  gap={same_pca - diff_pca:.4f}")
print()
print("A larger gap means the model better separates bonds by credit quality.")

In [ ]:
# Distribution of pairwise cosine similarities: same-rating vs. different-rating
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (sims, title) in zip(axes, [
    (s_qcml, f"QCML embeddings\ngap = {same_qcml - diff_qcml:.4f}"),
    (s_pca,  f"PCA projections\ngap = {same_pca - diff_pca:.4f}"),
]):
    ax.hist(sims[s_gt_r == 1], bins=50, density=True, alpha=0.6,
            color="#2196F3", label="Same rating")
    ax.hist(sims[s_gt_r == 0], bins=50, density=True, alpha=0.6,
            color="#F44336", label="Different rating")
    ax.axvline(same_qcml if "QCML" in title else same_pca,
               color="#2196F3", linestyle="--", linewidth=1.5)
    ax.axvline(diff_qcml if "QCML" in title else diff_pca,
               color="#F44336", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.25)

plt.suptitle("Bond Pairwise Similarity Distributions\n"
             "A clear separation means the model identifies comparable bonds",
             fontsize=11)
plt.tight_layout()
plt.savefig("bond_similarity_distributions.png", bbox_inches="tight", dpi=150)
plt.show()

## 10. Feature Diagnostics — Which Bond Attributes Drive QCML Geometry?

In [ ]:
eig_spread = model.observable_eigenvalue_spread()          # (D,)
q_var      = model.per_feature_quantum_variance(X)         # (D,)
recon_mse  = model.per_feature_recon_mse(X)                # (D,)

diag_df = pd.DataFrame({
    "feature"    : FEAT_NAMES,
    "eig_spread" : eig_spread,
    "q_variance" : q_var,
    "recon_mse"  : recon_mse,
}).sort_values("eig_spread", ascending=False).reset_index(drop=True)

print("Bond features ranked by QCML observable eigenvalue spread:")
print(diag_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics   = ["eig_spread", "q_variance", "recon_mse"]
titles    = [
    "Observable eigenvalue spread\n(active encoding)",
    "Quantum variance per feature\n(uncertainty)",
    "Reconstruction MSE per feature\n(fit quality)",
]

for ax, m, t in zip(axes, metrics, titles):
    sub = diag_df.sort_values(m, ascending=(m == "recon_mse"))
    colors = ["#1565C0" if v >= sub[m].median() else "#90CAF9" for v in sub[m]]
    ax.barh(sub["feature"], sub[m], color=colors, edgecolor="white")
    ax.set_title(t, fontsize=9)
    ax.grid(True, axis="x", alpha=0.3)

plt.suptitle("QCML Bond Feature Diagnostics", fontsize=11)
plt.tight_layout()
plt.savefig("bond_feature_diagnostics.png", bbox_inches="tight", dpi=150)
plt.show()

## 11. Spectral Gap Analysis — Identifying Borderline Bonds

The spectral gap $\lambda_1 - \lambda_0$ measures ground-state robustness.

**In bond markets this has a natural interpretation:**  
A bond at the BB/B boundary (e.g., a fallen angel or a rising star) has a credit profile that sits near a boundary in the learned geometry. Its ground state is nearly degenerate — a small move in fundamentals (leverage, coverage) could tip it from one cluster to another. These are exactly the bonds that require analyst attention for rating change risk.

In [ ]:
print("Computing spectral gaps (sample of 300 bonds)...")
idx_gap  = rng.choice(N_TOTAL, size=300, replace=False)
gaps     = model.spectral_gaps(X[idx_gap], n_samples=300)
y_r_gap  = y_rating[idx_gap]

gap_by_rating = {r: gaps[y_r_gap == r] for r in range(3)}

print("Mean spectral gap by rating:")
for r, g in gap_by_rating.items():
    print(f"  {RATING_NAMES[r]:4s}: mean={g.mean():.4f}  std={g.std():.4f}  "
          f"(n={len(g)})")

# Bottom 10% of gaps — most uncertain bonds
threshold = np.percentile(gaps, 10)
borderline_idx = np.where(gaps < threshold)[0]
borderline_ratings = y_r_gap[borderline_idx]
print(f"\nBottom 10% spectral gap (n={len(borderline_idx)} bonds):")
print(f"  Rating composition: ", dict(zip(*np.unique(borderline_ratings, return_counts=True))))

## 12. Visualisations

In [ ]:
fig = plt.figure(figsize=(16, 13))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)

# ── Panel 1: Training loss ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(model.loss_history, color="#1565C0", linewidth=1.4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("QCML Training Loss")
ax1.grid(True, alpha=0.25)

# ── Panel 2: PCA of Z coloured by rating ────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
pca2 = PCA(n_components=2, random_state=SEED)
Z2   = pca2.fit_transform(Z)
for r in range(3):
    mask = y_rating == r
    ax2.scatter(Z2[mask, 0], Z2[mask, 1], c=RATING_COLORS[r],
                label=RATING_NAMES[r], s=14, alpha=0.65, edgecolors="none")
ax2.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
ax2.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
ax2.set_title("QCML Embeddings — Coloured by Rating")
ax2.legend(markerscale=2, fontsize=8)
ax2.grid(True, alpha=0.2)

# ── Panel 3: PCA of Z coloured by sector ────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
for s in range(N_SECTORS):
    mask = y_sector == s
    ax3.scatter(Z2[mask, 0], Z2[mask, 1], c=SECTOR_COLORS[s],
                label=SECTOR_NAMES[s], s=14, alpha=0.65,
                marker=["o","s","^","D","v"][s], edgecolors="none")
ax3.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
ax3.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
ax3.set_title("QCML Embeddings — Coloured by Sector")
ax3.legend(markerscale=2, fontsize=7, ncol=2)
ax3.grid(True, alpha=0.2)

# ── Panel 4: Spectral gap by rating ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for r in range(3):
    ax4.hist(gap_by_rating[r], bins=30, density=True,
             color=RATING_COLORS[r], alpha=0.6, label=RATING_NAMES[r])
ax4.axvline(threshold, color="black", linestyle=":", linewidth=1.5,
            label=f"10th pct (borderline)")
ax4.set_xlabel("Spectral gap  λ₁ − λ₀")
ax4.set_ylabel("Density")
ax4.set_title("Spectral Gap by Rating\n(low gap = borderline bond = analyst review needed)")
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.25)

# ── Panel 5: Similarity recovery bar chart ──────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
methods = ["QCML", "PCA"]
same_vals = [same_qcml, same_pca]
diff_vals = [diff_qcml, diff_pca]
x = np.arange(len(methods))
w = 0.35
ax5.bar(x - w/2, same_vals, w, label="Same-rating pairs", color="#2196F3", alpha=0.85)
ax5.bar(x + w/2, diff_vals, w, label="Diff-rating pairs", color="#F44336", alpha=0.85)
ax5.set_xticks(x); ax5.set_xticklabels(methods)
ax5.set_ylabel("Mean cosine similarity")
ax5.set_title("Similarity Recovery: Same vs. Different Rating\n"
              "(larger gap = better bond comparables ranking)")
ax5.legend(fontsize=8)
ax5.grid(True, axis="y", alpha=0.3)

# ── Panel 6: Reconstruction errors by rating ────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
errs = model.reconstruction_errors(X)
for r in range(3):
    ax6.hist(errs[y_rating == r], bins=30, density=True,
             color=RATING_COLORS[r], alpha=0.6, label=RATING_NAMES[r])
ax6.set_xlabel("Reconstruction MSE")
ax6.set_ylabel("Density")
ax6.set_title("QCML Reconstruction Error by Rating\n(CCC bonds harder to reconstruct = more idiosyncratic)")
ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.25)

plt.suptitle("QCML High-Yield Bond Clustering — Summary", fontsize=13)
plt.savefig("qcml_bond_summary.png", bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved: qcml_bond_summary.png")

## 13. Hilbert Dimension Sweep

In [ ]:
sweep = []
for m in [8, 12, 16, 24]:
    qm = QCML(hilbert_dim=m, lr=LR, w=W, seed=SEED)
    qm.fit(X, epochs=400, batch_size=BATCH, verbose=False)
    Zm   = qm.transform(X)
    km_m = best_kmeans(Zm, K_RATING)
    acc_m = cluster_accuracy(y_rating, km_m, K_RATING)
    ari_m = adjusted_rand_score(y_rating, km_m)
    # Similarity gap
    Zs  = Zm[idx]
    sim_m = Zs @ Zs.T
    gap_m = sim_m[iu][s_gt_r == 1].mean() - sim_m[iu][s_gt_r == 0].mean()
    sweep.append({"m": m, "Accuracy": round(acc_m, 4), "ARI": round(ari_m, 4),
                  "Similarity_gap": round(gap_m, 4)})
    print(f"  m={m:2d}: Acc={acc_m:.4f}  ARI={ari_m:.4f}  SimGap={gap_m:.4f}")

sweep_df = pd.DataFrame(sweep)
print()
print(sweep_df.to_string(index=False))

## 14. Results Summary

In [ ]:
summary = pd.DataFrame([
    {
        "Method"         : "K-Means — raw features",
        "Rating Acc"     : round(acc_raw_r, 4),
        "Rating ARI"     : round(met_raw_r["ARI"], 4),
        "Rating NMI"     : round(met_raw_r["NMI"], 4),
        "Sim Gap"        : "—",
    },
    {
        "Method"         : f"PCA ({pca.n_components_} components) + K-Means",
        "Rating Acc"     : round(acc_pca_r, 4),
        "Rating ARI"     : round(met_pca_r["ARI"], 4),
        "Rating NMI"     : round(met_pca_r["NMI"], 4),
        "Sim Gap"        : round(same_pca - diff_pca, 4),
    },
    {
        "Method"         : f"QCML (m={HILBERT_DIM}, w={W}) + K-Means",
        "Rating Acc"     : round(acc_qcml_r, 4),
        "Rating ARI"     : round(met_qcml_r["ARI"], 4),
        "Rating NMI"     : round(met_qcml_r["NMI"], 4),
        "Sim Gap"        : round(same_qcml - diff_qcml, 4),
    },
])

print("Similarity Gap = mean cosine sim (same-rating) − mean cosine sim (different-rating)")
print("Larger gap = better bond comparables identification\n")
summary.set_index("Method")

## 15. Discussion

**What QCML learns about the high-yield bond space:**

The ground-state embedding $|\psi_0(x)\rangle$ encodes the entire credit profile of a bond — spread, duration, leverage, coverage, liquidity — as a point on the unit sphere $S^{m-1}$. Bonds that are "similar" in fundamental credit terms should land in nearby regions of this sphere.

The similarity recovery analysis shows that QCML achieves a larger gap between same-rating and different-rating cosine similarities than PCA. This matters in practice because bond pricing by comparables works by finding the $k$ nearest neighbors in embedding space — a larger gap means cleaner comparable bond identification.

**The spectral gap as a rating-change signal:**
Bonds near the BB/B boundary have small spectral gaps — their ground state is nearly degenerate between two regions of the sphere. In a live portfolio management context, these are the *fallen angel candidates* (BB bonds at risk of dropping to B) and *rising stars* (B bonds improving toward BB). The spectral gap provides a model-intrinsic uncertainty flag at no additional computational cost.

**Observable eigenvalue spread and credit analysis:**
The feature importance ranking from QCML's `observable_eigenvalue_spread()` reveals which bond characteristics the algorithm found most structurally relevant. If leverage and spread dominate (as expected for HY), this validates the model's credit intuition. If an unexpected feature ranks highly (e.g., bid-ask spread), it may reveal that liquidity is acting as a proxy for something else — worth investigating.

**Connection to [Q3]:**
Rosaler et al. use a *supervised* variant where a target similarity matrix is provided as a training signal. This notebook uses the *unsupervised* QCML (no labels during training) and shows that the recovered geometry still reflects credit quality structure. The supervised extension would additionally train the observables to match a provided pairwise similarity target, which is the next natural step.

**Extensions:**
- Apply to real TRACE/WRDS bond transaction data with actual spreads and fundamental data
- Build a k-nearest-neighbor bond pricing model: for each bond with no recent trade, find its 5 nearest QCML neighbors and average their observed yields
- Extend to the supervised QCML variant from [Q3] using analyst-defined similarity labels
- Combine with QCML financial forecasting [Q2] for a joint pricing + forecasting pipeline
